In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="5" color="black">ch14_웹 데이터 수집_2</font>
***
# 1. Selenium
- [selenium](https://selenium-python.readthedocs.io/)을 이용한 동적 웹크롤링
- `pip install selenium`

* selenium - 4.47.0 / requests - 2.34.2 / urllib3 - 2.7.0

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

In [ ]:
dv = webdriver.Chrome() #새로운 크롬창 생성
dv.get("http://www.python.org") # 생성한 창에 url 삽입

In [ ]:
# 페이지 제목에 "Python"이라는 단어가 포함되어 있는지 확인
assert "Python" in dv.title 

In [ ]:
# 검색창 입력 필드(name="q") 검색 - By.CLASS_NAME, By.ID, By.CSS_SELETOR, By.TAG_NAME 등
el = dv.find_element(By.NAME, "q")

In [ ]:
el.clear() # 입력 필드 clear
el.send_keys("pycon") # input box에 pycon 삽입
el.send_keys(Keys.RETURN) # 엔터(Return) 입력으로 검색 시행

In [ ]:
el.send_keys(Keys.CONTROL, "a") # ctrl + a 입력 (입력 필드 내 전체 선택)
el.send_keys('pycon')
btn_el = dv.find_element(By.CSS_SELECTOR, 'button#submit')
btn_el.click()

In [ ]:
result_list = dv.find_elements(By.CSS_SELECTOR, 'li>h3>a')
for result in result_list:
    print('{} - {}'.format(result.text, result.get_attribute('href')))

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
for result in result_list:
    print('{} - https://www.python.org{}'.format(result.text, result.attrs.get('href')))

In [ ]:
from urllib.parse import urlparse
now_url = dv.current_url
print('현재 url :', now_url)
parsedUrl = urlparse(now_url)
print('파싱 url :', parsedUrl)
domain = f'{parsedUrl.scheme}://{parsedUrl.netloc}'
print('도메인 :', domain)

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(dv.page_source, 'html.parser')
result_list = soup.select('li>h3>a')
for result in result_list:
    print('{} - {}'.format(result.text, domain+result.attrs.get('href')))

In [ ]:
dv.close() # 창 제거

# 2. 동적 웹데이터 수집
- 동적 웹크롤링

## 2.1 다음 뉴스 검색

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

dv = webdriver.Chrome()
dv.get("http://www.daum.net")
time.sleep(0.5) # 페이지 로드 완료까지 0.5초 대기

word = input('검색할 단어 : ')

el = dv.find_element(By.NAME, "q")
el.send_keys(word)
el.send_keys(Keys.RETURN)

time.sleep(1)

dv.find_element(By.LINK_TEXT, "뉴스").click()

In [ ]:
# Soup 객체 사용 x
bodies = dv.find_elements(By.CSS_SELECTOR, 'div.item-title > strong.tit-g.clamp-g')

new_list = []

for body in bodies:
    a = body.find_element(By.TAG_NAME, 'a')
    title = a.text
    link = a.get_attribute('href')
    new_list.append([title, link])

In [ ]:
# 페이지 이동
page_nav = dv.find_element(By.CLASS_NAME, 'inner_paging')
next_page = page_nav.find_element(By.LINK_TEXT, "3")
next_page.click()

In [ ]:
# Soup 객체 사용 o

import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse

url = dv.current_url

response = requests.get(url)

soup = BeautifulSoup(response.text, 'html.parser')

news_list = []

news_el = soup.select('div.item-title > strong.tit-g.clamp-g > a')

for idx, item in enumerate(news_el):
    news_list.append({'no' : idx+1, 'title' : item.text, 'link' : item.attrs.get('href')})
    
rslt = news_list
    
import pandas as pd
pd.DataFrame(rslt)

In [ ]:
dv.quit()

## 2.2 다음 뉴스 페이징 처리

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

dv = webdriver.Chrome()
dv.get("http://www.daum.net")
time.sleep(0.5) # 페이지 로드 완료까지 0.5초 대기

word = input('검색할 단어 : ')

el = dv.find_element(By.NAME, "q")
el.send_keys(word)
el.send_keys(Keys.RETURN)

time.sleep(1)

dv.find_element(By.LINK_TEXT, "뉴스").click()

pages = int(input('수집할 페이지 수 : '))
time.sleep(1)

new_list = []

for page in range(1, 1+pages):
    
    bodies = dv.find_elements(By.CSS_SELECTOR, 'div.item-title > strong.tit-g.clamp-g')
    print('수집 중인 페이지 : ', page)
    for body in bodies:
        a = body.find_element(By.TAG_NAME, 'a')
        title = a.text
        link = a.get_attribute('href')
        new_list.append([title, link])
        
    time.sleep(3)
    
    page_nav = dv.find_element(By.CLASS_NAME, 'inner_paging')
    next_page = page_nav.find_element(By.LINK_TEXT, str(page+1))
    next_page.click()
    
news_df = pd.DataFrame(new_list, columns=['title', 'link'])
print('수집 완료 : news_df', news_df.shape)
display(news_df.head())

In [ ]:
dv.quit()

## 2.3 맞춤법 검사기
- 네이버 맞춤법 검사기 활용

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time

In [ ]:
dv = webdriver.Chrome()
dv.get('https://www.naver.com')
time.sleep(0.5)

el = dv.find_element(By.NAME, "query")
el.clear()
el.send_keys('맞춤법검사기')
el.send_keys(Keys.RETURN)
time.sleep(0.5)

chk = dv.find_element(By.CLASS_NAME, "txt_gray")
para = input('검사할 문장 : ')
chk.send_keys(para)
dv.find_element(By.CLASS_NAME, "btn_check").click()
time.sleep(0.5)

rslt = dv.find_element(By.CLASS_NAME, "_result_text")
print('맞춤법 교정 결과 : ', rslt.text)
dv.quit()

In [ ]:
with open('data/ch14_맞춤법검사_전.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()

ready_text_list = [] # 300자 내외의 문장 단위로 나눠진 텍스트의 list

while len(text)>=300:
    print(len(text))
    temp = text[:300]
    last_dot_index = temp.rfind('\n')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)

In [ ]:
print([read_text for read_text in ready_text_list])

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time

In [ ]:
dv = webdriver.Chrome()
dv.get('https://www.naver.com')
time.sleep(0.5)

el = dv.find_element(By.NAME, "query")
el.send_keys('맞춤법검사기')
el.send_keys(Keys.RETURN)
time.sleep(1)

chk = dv.find_element(By.CLASS_NAME, "txt_gray")
rslts = ''

for idx, ready_text in enumerate(ready_text_list):
    print(f'{idx+1}/{len(ready_text_list)}')
    chk.clear()
    chk.send_keys(ready_text)
    dv.find_element(By.CLASS_NAME, "btn_check").click()
    time.sleep(2)
    rslt = dv.find_element(By.CLASS_NAME, "_result_text").text
    rslts += rslt + '\n'


print('맞춤법 교정 완료')
dv.quit()

In [ ]:
with open('data/ch14_맞춤법검사_후.txt', 'w', encoding='utf-8') as f:
    f.write(rslts)

## quiz

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

with open('data/ch14_맞춤법후.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()

dv = webdriver.Chrome()
dv.get('https://papago.naver.com/')

print('번역기 접속 완료')
time.sleep(1)

close_box = dv.find_element(By.CLASS_NAME, "entry-popup-module-scss-module__UZIxta__close")
close_box.click()

print('파일 번역 진행 중 ...')
time.sleep(1)

chk = dv.find_element(By.CSS_SELECTOR,\
                      "div.text-translator-module-scss-module__CYJRkW__original-textarea > \
                      div.text-editor-module-scss-module__gKzuvW__text-area.text-editor-module-scss-module__gKzuvW__dynamic-md.text-translator-module-scss-module__CYJRkW__text-editor")

chk.send_keys(text)
time.sleep(5)
rslt_list = dv.find_elements(By.CSS_SELECTOR, "div.text-translator-module-scss-module__CYJRkW__has-translation div.text-editor-module-scss-module__gKzuvW__text-area > p > span")

rslts = ''

for rslt in rslt_list:
    rslts += rslt.text + '\n\n'

print('한 → 영 번역 완료')
dv.quit()

with open('data/ch14_자동화영어번역본.txt', 'w', encoding='utf-8') as f:
    f.write(rslts)

- 3000자에 맞춰 파일 정리 및 번역본 출력

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import time

# 파일 불러오기
with open('data/ch14_맞춤법후.txt', 'r', encoding='utf-8') as fp:
    text = fp.read()

# 프로그램 규격에 맞추어 파일 정리
ready_text_list = []

while len(text)>=3000:
    print(len(text))
    temp = text[:3000]
    last_dot_index = temp.rfind('\n')
    ready_text_list.append(text[:last_dot_index+1])
    text = text[last_dot_index+1:]
ready_text_list.append(text)
    
# 번역기 접속
dv = webdriver.Chrome()
dv.get('https://papago.naver.com/')

print('번역기 접속 완료')
time.sleep(1)

# 팝업 닫기
close_box = dv.find_element(By.CLASS_NAME, "entry-popup-module-scss-module__UZIxta__close")
if close_box:
    close_box.click()

print('파일 번역 진행 중 ...')
time.sleep(1)

# 텍스트 입력 위치 지정
chk = dv.find_element(By.CSS_SELECTOR,\
                      "div.text-translator-module-scss-module__CYJRkW__original-textarea > \
                      div.text-editor-module-scss-module__gKzuvW__text-area.text-editor-module-scss-module__gKzuvW__dynamic-md.text-translator-module-scss-module__CYJRkW__text-editor")

# 텍스트 입력
rslts = ''

for idx, ready_text in enumerate(ready_text_list):
    print(f'{idx+1}/{len(ready_text_list)}')
    chk.send_keys(Keys.CONTROL, "a")
    chk.send_keys(ready_text)
    time.sleep(5)
    rslt = dv.find_elements(By.CSS_SELECTOR, 'div[class^="text-editor-module-scss-module"]')[1].text
    rslts += rslt + '\n\n'



print('한 → 영 번역 완료')
dv.quit()

with open('data/ch14_자동화영어번역본.txt', 'w', encoding='utf-8') as f:
    f.write(rslts)